In [1]:
import numpy as np
import sklearn as sk
import pandas as pd
import data_management as dm
import feature_builders as fb
import tree_models as tm
import model_wrappers as wp
from graph_helpers import plot_var_vs_loss, plot_ewma_bank

In [2]:
TICKERS = ["AAPL", "JPM", "XOM", "KO"]

WEIGHTS = {
    "AAPL": 0.25,
    "JPM": 0.25,
    "XOM": 0.25,
    "KO": 0.25,
}

df = pd.read_csv(
    "./data_general/4_ticker_data.csv",
    parse_dates=["Date"],
    index_col="Date",
)


tickers_data = df[TICKERS].pct_change().dropna()



In [3]:
calendar_last_180_days = dm.DailyGroundTruthCalendar(
    df, TICKERS, WEIGHTS,
    train_start="2024-06-03",   # Pazartesi, ~6 ay geriye (180 takvim günü civarı)
    train_end="2024-11-29",     # Cuma, Kasım'ın son iş günü
    forecast_start="2024-12-02",  # Pazartesi
    forecast_end="2024-12-31",
)

[WARNING] forecast_end=2024-12-31 falls on Tuesday, expected Friday (using it as-is).
[WARNING] [train] 4 weekday days had no data and were skipped (likely market holidays): 2024-06-19, 2024-07-04, 2024-09-02, 2024-11-28
[INFO] train set: 126 bars, 4 weekday days skipped
[WARNING] [forecast] 1 weekday day had no data and were skipped (likely market holidays): 2024-12-25
[INFO] forecast set: 21 bars, 1 weekday day skipped


In [4]:
training_feature_long, forecast_feature_long = fb.build_long_features(calendar_last_180_days, target_scale= 10000)

[INFO] long train: 504 rows -> 424 after dropna (80 dropped, 15.9%)
[INFO] long forecast: 88 rows -> 88 after dropna (0 dropped, 0.0%)


In [5]:
VAR_CONFIDENCE = 0.95

# Both banks are SINGLE-USE: training fills them in place and marks them trained, so they cannot be given to
# another training run (create new banks per run; creating them in this same cell keeps re-running it safe).
residual_bank = dm.ResidualBank(
    mu_mode="constant",                         # "constant": mu_hat = default_mu_hat | "rolling": mean of the last rolling_mu_hat_window_size returns
    default_mu_hat=0.0,
    rolling_mu_hat_window_size=50,              # only read when mu_mode="rolling"
)

# built from the calendar: tickers and weights are read from it (training checks it is the same calendar)
ewma_bank = dm.EwmaVarianceBank(
    calendar_last_180_days,
    ewma_lam=0.94,                              # RiskMetrics decay: larger = forgets slower
    ewma_min_periods=30,                        # the first N train bars seed the covariance (EWMA burn-in)
)

model = tm.xgboost_train(calendar_last_180_days, training_feature_long, residual_bank=residual_bank, ewma_bank=ewma_bank)

[INFO] train_dates_past_burn_in: 96/105 candidate dates are past the burn-in (burn_in_bars=29, warmup_bars=0, train_data starts 2024-06-03, burn-in ends 2024-07-16, 9 date(s) dropped).


In [6]:
# fills copies of the model's two banks with the forecast window (no VaR yet, nothing about reading involved)
residualbank, ewmabank = model.calculate_multi_day_standart_residuals_from_long_format_model(
    calendar=calendar_last_180_days,
    long_forecast_feature=forecast_feature_long,
)

# reads the VaR off the filled bank, adds the realized loss, records the run under run_key.
# Everything about HOW to read is chosen here, so the same filled bank can be read again any other way.
var_df = calendar_last_180_days.var_from_residual_bank_forecast_period(
    residualbank,
    var_confidence=VAR_CONFIDENCE,                  # e.g. 0.95 -> takes the 5% lower quantile of the residuals
    multi_day_var_calculation_window_size=360,      # int: use at most the last N residuals; None: use all residuals available
    run_key="xgboost_long",                         # key in calendar.var_prediction_runs (reusing a key overwrites that run)
    multi_day_var_min_periods=30,                   # min prior residuals a day needs to get a VaR (else ValueError)
)

In [7]:
var_df

,varForecast,n_residuals_used,loss
Date,,,
2024-12-02,0.006547,96,0.003004
2024-12-03,0.00713,97,-0.000849
2024-12-04,0.006436,98,0.013437
2024-12-05,0.007342,99,-0.006114
2024-12-06,0.007982,100,0.002476
2024-12-09,0.008419,101,0.000758
2024-12-10,0.008653,102,-0.000788
2024-12-11,0.009347,103,0.003339
2024-12-12,0.009498,104,-0.004004


In [8]:
plot_var_vs_loss(var_df, residualbank, calendar=calendar_last_180_days)   # calendar verilirse train/forecast bantlari cizilir

In [9]:
# EwmaVarianceBank.portfolio_variance: her gunun portfoy varyansi (bank'in asil ciktisi, ResidualBank'a gider)
# Egitim (96 gun) + forecast (21 gun) = 117 satir; indeks = sonucun yasandigi gun
ewmabank.portfolio_variance.head()

2024-07-17    0.000043
2024-07-18    0.000056
2024-07-19    0.000032
2024-07-22    0.000031
2024-07-23    0.000030
Name: predicted_variance, dtype: float64

In [10]:
ewmabank.portfolio_variance.tail()

2024-12-24    0.000081
2024-12-26    0.000056
2024-12-27    0.000046
2024-12-30    0.000057
2024-12-31    0.000056
Name: predicted_variance, dtype: float64

In [11]:
ewmabank.cov_by_date

{Timestamp('2024-07-17 00:00:00'):           AAPL       JPM       XOM        KO
 AAPL  0.000355 -0.000071 -0.000072 -0.000013
 JPM  -0.000071  0.000145  0.000054 -0.000015
 XOM  -0.000072  0.000054  0.000136  0.000015
 KO   -0.000013 -0.000015  0.000015  0.000053,
 Timestamp('2024-07-18 00:00:00'):           AAPL           JPM       XOM            KO
 AAPL  0.000372 -8.964930e-05 -0.000089 -3.423713e-05
 JPM  -0.000090  1.504149e-04  0.000063 -5.344926e-07
 XOM  -0.000089  6.342046e-05  0.000139  2.632065e-05
 KO   -0.000034 -5.344926e-07  0.000026  6.232949e-05,
 Timestamp('2024-07-19 00:00:00'):           AAPL           JPM       XOM            KO
 AAPL  0.000375 -4.512625e-05 -0.000095 -3.180506e-05
 JPM  -0.000045  2.019506e-04  0.000041  8.214413e-08
 XOM  -0.000095  4.081864e-05  0.000137  2.455998e-05
 KO   -0.000032  8.214413e-08  0.000025  5.859536e-05,
 Timestamp('2024-07-22 00:00:00'):           AAPL           JPM       XOM            KO
 AAPL  0.000353 -4.245182e-05 -0.0000

In [12]:
# kac gunun kovaryansi kayitli ve son gunun matrisi (matrisler ticker etiketli DataFrame)
print(len(ewmabank.cov_by_date), "gun")
last_date = ewmabank.portfolio_variance.index[-1]
ewmabank.cov_by_date[last_date]

117 gun


,AAPL,JPM,XOM,KO
AAPL,0.000117,0.000048,0.000003,0.000028
JPM,0.000048,0.000234,0.000044,0.000001
XOM,0.000003,0.000044,0.000095,0.000033
KO,0.000028,0.000001,0.000033,0.000071


In [13]:
plot_ewma_bank(ewmabank)                                                  # varsayılan


In [14]:
plot_ewma_bank(ewmabank, split_date=calendar_last_180_days.train_data_last_index[0])   # eğitim/forecast sınırı


In [15]:
plot_ewma_bank(ewmabank, as_volatility=True)                              # sqrt ve % olarak


In [28]:
# bank'in train gunleri icin VaR (in-sample); ilk multi_day_var_min_periods gun onceki residual olmadigi icin dusulur
train_var = calendar_last_180_days.var_from_residual_bank_train_period(
    residualbank,
    var_confidence=VAR_CONFIDENCE,
    multi_day_var_calculation_window_size=10,
    multi_day_var_min_periods=10,
)
train_var

,varForecast,n_residuals_used,loss
Date,,,
2024-07-31,0.009581,10,0.001613
2024-08-01,0.008479,10,0.008761
2024-08-02,0.010464,10,0.003996
2024-08-05,0.007572,10,0.026313
2024-08-06,0.021523,10,-0.003029
...,...,...,...
2024-11-22,0.005777,10,-0.005688
2024-11-25,0.005243,10,-0.003076
2024-11-26,0.004324,10,0.001476


In [29]:
plot_var_vs_loss(train_var, residualbank, calendar=calendar_last_180_days)

In [18]:
# train + forecast birlikte (bank'in tum gunleri); ilk multi_day_var_min_periods gun yine dusulur, run kaydedilmez
entire_var = calendar_last_180_days.var_from_residual_bank_entire_period(
    residualbank,
    var_confidence=VAR_CONFIDENCE,
    multi_day_var_calculation_window_size=60,
    multi_day_var_min_periods=30,
)
plot_var_vs_loss(entire_var, residualbank, title="VaR vs Realized Loss (train + forecast)", calendar=calendar_last_180_days)

## Wide format

Tek bir model, doğrudan **portföy** varyansını tahmin eder (ticker bazlı tahmin ve EWMA bank yok). Aynı calendar kullanılır, bank'lar ve değişkenler long bölümünden ayrı adlandırılmıştır (`*_wide`).

In [19]:
training_feature_wide, forecast_feature_wide = fb.build_wide_features(calendar_last_180_days, target_scale=10000)

[INFO] wide train: 126 bars -> 106 after dropna (20 dropped, 15.9%)
[INFO] wide forecast: 22 bars -> 22 after dropna (0 dropped, 0.0%)


In [20]:
# residual bank da tek kullanimlik: her egitim icin yeni bir tane (wide yolda EwmaVarianceBank yok)
residual_bank_wide = dm.ResidualBank(
    mu_mode="constant",
    default_mu_hat=0.0,
    rolling_mu_hat_window_size=50,              # only read when mu_mode="rolling"
)

model_wide = tm.xgboost_train(calendar_last_180_days, training_feature_wide, residual_bank=residual_bank_wide)

In [21]:
# forecast penceresini model bank'inin kopyasina yazar (VaR yok), sonra VaR bank'tan okunur
residualbank_wide = model_wide.calculate_multi_day_standart_residuals_from_wide_format_model(
    calendar=calendar_last_180_days,
    forecast_feature=forecast_feature_wide,
)

var_df_wide = calendar_last_180_days.var_from_residual_bank_forecast_period(
    residualbank_wide,
    var_confidence=VAR_CONFIDENCE,
    multi_day_var_calculation_window_size=360,      # int: use at most the last N residuals; None: use all residuals available
    run_key="xgboost_wide",
    multi_day_var_min_periods=30,
)
var_df_wide.head()

,varForecast,n_residuals_used,loss
Date,,,
2024-12-02,0.006498,105,0.003004
2024-12-03,0.010308,106,-0.000849
2024-12-04,0.010639,107,0.013437
2024-12-05,0.007693,108,-0.006114
2024-12-06,0.012089,109,0.002476


In [22]:
plot_var_vs_loss(var_df_wide, residualbank_wide, calendar=calendar_last_180_days)

## Modelleri karşılaştırma (aynı calendar)

Long ve wide, aynı `calendar_last_180_days` üzerinde çalıştı. Aşağıda VaR'ları ve model çıktılarını (predicted_variance) tek grafikte, gerçekleşen EWMA varyansıyla birlikte görüyoruz.

In [23]:
from graph_helpers import plot_var_models_vs_loss, plot_models_output_vs_realized_variance, plot_model_output_vs_realized_variance

# {model_adi: (var_df, residualbank)}, hepsi ayni calendar'dan olmali (loss kolonlari tutarsizsa hata verir)
plot_var_models_vs_loss({
    "xgboost_long": (var_df, residualbank),
    "xgboost_wide": (var_df_wide, residualbank_wide),
}, calendar=calendar_last_180_days)

In [24]:
# gerceklesen getirilerle doldurulan EWMA bank (model karisik degil): warmup ile isitilir, sonra train + forecast eklenir.
# bank'in d gunu degeri d-1'e kadar olan getirilerden hesaplanir (d'nin kendi getirisi dahil degil)
realized_bank = calendar_last_180_days.ewma_realized_variance_bank(ewma_lam=0.94, ewma_min_periods=30)

# gercek realized variance: d gununun degeri d'nin kendi getirisini (r_d) icerir = bank'in bir sonraki bar degeri
realized_variance = calendar_last_180_days.ewma_realized_variance(ewma_lam=0.94, ewma_min_periods=30)
realized_variance.head()

Date
2024-07-16    0.000030
2024-07-17    0.000030
2024-07-18    0.000035
2024-07-19    0.000035
2024-07-22    0.000033
Name: ewma_realized_variance, dtype: float64

In [25]:
# tek model: model_output vs gerceklesen EWMA varyans (sadece forecast penceresi)
plot_model_output_vs_realized_variance(calendar_last_180_days, residualbank, model_name="xgboost_long")

In [26]:
# birden fazla model: ayni calendar, tum bank tarihleri (egitim in-sample + forecast)
plot_models_output_vs_realized_variance(
    calendar_last_180_days,
    {"xgboost_long": residualbank, "xgboost_wide": residualbank_wide},
    only_forecast_window=False,     # True: sadece forecast penceresi
)